# 11 - Qwen3 Embedding 8B Index and Retrieval Evaluation

Heavy embedding ablation. This builds a separate dense index with `Qwen/Qwen3-Embedding-8B` and evaluates it on the locked `gold_benchmark_v1.csv` benchmark. The official-law corpus is unchanged.

In [ ]:
!pip install -q -U "sentence-transformers>=5.1.0" "transformers>=4.51.0" accelerate faiss-cpu rank-bm25

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import sys
import torch

DRIVE_ROOT = Path('/content/drive/MyDrive/rag')
sys.path.insert(0, str(DRIVE_ROOT))
sys.path.insert(0, str(DRIVE_ROOT / 'src'))

config = json.loads((DRIVE_ROOT / 'project_config.json').read_text(encoding='utf-8'))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
from src.build_index import build_indexes

embedding_model = 'Qwen/Qwen3-Embedding-8B'
corpus_csv = DRIVE_ROOT / config['main_law_corpus_csv']
index_root = DRIVE_ROOT / 'indexes/official_law_v3_qwen3_embedding_8b'

# 8B embedding is intentionally heavy. If Colab OOMs, lower batch_size to 1.
manifest = build_indexes(
    corpus_path=corpus_csv,
    index_root=index_root,
    embedding_model=embedding_model,
    text_field=config['retrieval_text_field'],
    batch_size=2,
    device=device,
    build_dense=True,
    build_bm25=False,
)
manifest

In [ ]:
from src.evaluation_retrieval import evaluate_retrieval

benchmark_csv = DRIVE_ROOT / config['benchmark_csv']
summary = evaluate_retrieval(
    benchmark_csv=benchmark_csv,
    index_root=index_root,
    output_predictions_csv=DRIVE_ROOT / 'outputs/retrieval_eval/qwen3_embedding_8b_dense_predictions_v1.csv',
    output_summary_json=DRIVE_ROOT / 'outputs/retrieval_eval/qwen3_embedding_8b_dense_summary_v1.json',
    mode='dense',
    top_k=30,
    candidate_k=30,
    device=device,
)
summary

In [ ]:
import pandas as pd

rows = []
bge_summary = DRIVE_ROOT / 'outputs/retrieval_eval/dense_retrieval_summary_v1.json'
if bge_summary.exists():
    old = json.loads(bge_summary.read_text(encoding='utf-8'))
    rows.append({'experiment': 'BGE-M3 dense', **old['metrics']})
rows.append({'experiment': 'Qwen3-Embedding-8B dense', **summary['metrics']})
pd.DataFrame(rows)[[
    'experiment', 'doc_hit@5', 'doc_hit@10', 'article_hit@5', 'article_hit@10',
    'doc_mrr', 'article_mrr', 'article_ndcg@5', 'article_ndcg@10'
]]